In [5]:
%run run_in_jupyter.py.ipynb

ModuleNotFoundError: No module named 'app'

ModuleNotFoundError: No module named 'app'

In [7]:
"""
app.py
Flask website: user enters pollutant values (PM2.5, PM10, NO2, SO2, O3, CO),
gets a predicted numeric US AQI value + category from the trained
AdaBoost regression model.

Run with:  python app.py
Then open: http://127.0.0.1:5000
"""

from flask import Flask, render_template, request
import joblib
import pandas as pd

app = Flask(__name__)

model = joblib.load("aqi_regressor.joblib")

FEATURES = ["PM2_5_ugm3", "PM10_ugm3", "NO2_ugm3", "SO2_ugm3", "O3_ugm3", "CO_ugm3"]


def aqi_category(aqi_value):
    """Map a numeric AQI value to its standard US EPA category + color."""
    if aqi_value <= 50:
        return "Good", "#00e400"
    elif aqi_value <= 100:
        return "Moderate", "#ffff00"
    elif aqi_value <= 150:
        return "Unhealthy for Sensitive Groups", "#ff7e00"
    elif aqi_value <= 200:
        return "Unhealthy", "#ff0000"
    elif aqi_value <= 300:
        return "Very Unhealthy", "#8f3f97"
    else:
        return "Hazardous", "#7e0023"


@app.route("/", methods=["GET"])
def home():
    return render_template("index.html", prediction=None)


@app.route("/predict", methods=["POST"])
def predict():
    try:
        values = [float(request.form[feat]) for feat in
                  ["PM2.5", "PM10", "NO2", "SO2", "O3", "CO"]]
        X_input = pd.DataFrame([values], columns=FEATURES)

        predicted_aqi = round(float(model.predict(X_input)[0]), 1)
        category, color = aqi_category(predicted_aqi)

        return render_template(
            "index.html",
            prediction=predicted_aqi,
            category=category,
            color=color,
        )
    except Exception as e:
        return render_template("index.html", prediction=f"Error: {e}")


if __name__ == "__main__":
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (windowsapi)


SystemExit: 1